# Ground-Truth-Evaluation: Modell vs. manuelle Bewertung

Reproduzierbarer Vergleich der Prototyp-Scores gegen eine manuell bewertete Referenzstichprobe.
Logik in `src/evaluation/ground_truth.py`, Konzept in `docs/methodik_evaluation_ground_truth`.

**Ablauf:**
1. Modell auf der Stichprobe scoren (Testergebnisse).
2. Gefülltes Bewertungs-Template (`ground_truth_template.csv`, 0–3 je Dimension) laden.
3. Mergen, Modell-Urteil per gleicher Regel ableiten.
4. Kennzahlen (Spearman, Kendall τ, Accuracy, Cohen's κ, gewichtetes κ, Konfusion).
5. Thesis-Grafiken erzeugen.

> **Vor dem Lauf:** die 0–3-Spalten (`gt_findability`, `gt_accessibility`, `gt_reusability`, `gt_expressiveness`) im Template ausfüllen — **blind** gegenüber den Modell-Scores.

In [ ]:
import sys
from pathlib import Path

# Repo-Root robust finden (egal von wo das Notebook startet).
REPO = Path.cwd()
while not (REPO / 'src').exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'src'))

from dotenv import load_dotenv
load_dotenv(REPO / '.env')

import pandas as pd
from IPython.display import Image, display
from evaluation import ground_truth as gt

# ===================== Parameter =====================
SAMPLE_DIR = REPO / 'data' / 'sample_2026-06-18_13-45'   # <- Stichproben-Verzeichnis
GT_CSV     = SAMPLE_DIR / 'ground_truth_template.csv'    # gefülltes Bewertungs-Template
FIG_DIR    = SAMPLE_DIR / 'eval_figures'
USE_LLM    = True    # expressiveness braucht das LLM (kostet Tokens!)
RECOMPUTE  = False   # True erzwingt Neuberechnung statt model_scores.csv-Cache
# =====================================================
print('REPO     :', REPO)
print('SAMPLE   :', SAMPLE_DIR, '|', len(list(SAMPLE_DIR.glob('*.rdf'))), 'RDF-Dateien')

## 1. Modell-Scores berechnen (mit Cache)

Bei `USE_LLM=True` wird je Datensatz ein LLM-Call für die Aussagekraft gemacht. Das Ergebnis wird in `model_scores.csv` gecacht — erneutes Ausführen lädt den Cache (kein erneuter Token-Verbrauch), außer `RECOMPUTE=True`.

In [ ]:
scores_csv = SAMPLE_DIR / 'model_scores.csv'
if scores_csv.exists() and not RECOMPUTE:
    model_df = pd.read_csv(scores_csv)
    print(f'Cache geladen: {scores_csv.name}')
else:
    llm = gt.make_llm() if USE_LLM else None
    print('LLM:', 'an' if llm else 'aus (expressiveness = NaN)')
    model_df = gt.run_model_scores(SAMPLE_DIR, llm=llm)
    model_df.to_csv(scores_csv, index=False)
    print('gespeichert:', scores_csv)
model_df.round(3)

## 2. Ground Truth laden

Leitet `gt_overall` (0–3-Mittel) und `gt_verdict` (gut/mittel/schlecht) aus den manuellen Bewertungen ab.

In [ ]:
gt_df = gt.load_ground_truth(GT_CSV)
n_rated = int(gt_df['gt_rated'].sum())
print(f'Vollständig bewertet: {n_rated}/{len(gt_df)} Datensätze')
gt_df[['file'] + gt.GT_COLS + ['gt_overall', 'gt_verdict']].head(10)

## 3. Merge & Vergleich

Modell-Dimensionsscores werden auf 0–3 reskaliert (×3) und mit **derselben** Urteilsregel zu einem Modell-Urteil verdichtet.

In [ ]:
merged = gt.merge_scores(gt_df, model_df)

if merged['gt_verdict'].notna().sum() == 0:
    print('Noch keine Bewertungen vorhanden — bitte die 0–3-Spalten im Template füllen und neu laufen lassen.')
    result = None
else:
    result = gt.compare(merged)
    print('=== Pro Dimension (Rangkorrelation Modell ↔ GT) ===')
    display(gt.summary_table(result))
    print('\n=== Gesamturteil ===')
    for k, v in result['overall'].items():
        print(f'  {k:18}: {round(v, 3) if isinstance(v, float) else v}')
    print('\n=== Konfusionsmatrix (Zeilen=GT, Spalten=Modell) ===')
    display(result['confusion'])

## 4. Grafiken (thesis-tauglich)

Werden als PNG in `eval_figures/` gespeichert und hier angezeigt.

In [ ]:
if result is not None:
    paths = gt.make_figures(merged, result, FIG_DIR)
    for p in paths:
        display(Image(str(p)))
    print('gespeichert:')
    for p in paths:
        print('  ', p)
else:
    print('Übersprungen — keine Bewertungen.')

## 5. Getrennt: Kernstichprobe vs. Extremfälle

Laut Methodik werden Kernstichprobe und Extremfälle **getrennt** ausgewertet. Dazu dieses Notebook ein zweites Mal mit `SAMPLE_DIR = REPO/'data'/'extreme_cases'` (bzw. dem Extremfall-Verzeichnis) laufen lassen — dort ebenfalls vorher ein `ground_truth_template.csv` via `scripts/build_ground_truth_template.py <dir>` erzeugen und bewerten.

Optional lassen sich beide Ergebnis-Tabellen am Ende nebeneinanderstellen (z. B. `pd.concat` der jeweiligen `summary_table`).